# 06 · Evaluation approaches

**Deck section 6** · slides 61–72

An end-to-end RAG score is useful and insufficient. It tells you something changed; it never
tells you what. This notebook builds the layered scorecard that does, then turns on the
component most teams trust without checking — the judge — and attacks it.

**By the end you can**

- read a symptom in production and name the metric that will move and the one that will not
- calibrate a judge with Cohen's κ and say why raw agreement is not enough
- catch judge drift, verbosity bias and position bias with probes you can run in CI
- execute the release-gate tree against a real candidate change and defend the verdict


In [ ]:
import pathlib, sys
ROOT = pathlib.Path.cwd()
while not (ROOT / "raglab" / "__init__.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from raglab.bootstrap import bootstrap
bootstrap(verbose=False)

import numpy as np
import pandas as pd
import raglab
from raglab import (viz, tables, catalog, generate, judge, metrics, pipeline, retrieve)
viz.reset_figures("6."); tables.reset_tables("6.")

bundle, index, pipe = raglab.quickstart(**raglab.TUNED)
rows = pipeline.evaluate(pipe, bundle.questions, pipe.chunks, personas=bundle.personas)
answerable = [r for r in rows if not r["is_null"]]
print(f"baseline evaluated: {len(rows)} questions")

---

## 6.1 Layered evaluation

Same examples, three scorecards. Use them to evaluate components independently *and* to
measure the final user-facing outcome — the two are different jobs and neither replaces the
other.


In [ ]:
viz.hld([
    dict(name="Retrieval", tone="index", nodes=[
        ("Recall@N", "the ceiling stage one buys"),
        ("Evidence Recall@k", "what reached the model"),
        ("Full-chain recall", "every hop arrived"),
        ("nDCG@k · MRR", "was it ranked well")]),
    dict(name="Context", tone="query", nodes=[
        ("Context precision", "share of packed slots carrying gold"),
        ("Provenance", "citations resolve to a packed chunk"),
        ("Ordering", "position sensitivity, measured"),
        ("Token utilisation", "what the cap actually bound")]),
    dict(name="Answer", tone="control", nodes=[
        ("Correctness", "against the gold answer"),
        ("Faithfulness", "every claim maps to a cited span"),
        ("Completeness", "every part of the question resolved"),
        ("Abstention", "precision and recall on the null set")]),
], title="Layered evaluation for RAG", kicker="Section 6 · structure",
   caption="Gold evidence and gold answer feed all three lanes. A regression that appears in "
           "one lane and not the others has just told you which stage moved.",
   source="Deck slide 62")

In [ ]:
def scorecard(rs):
    s = metrics.summarize(rs)
    ans = [r for r in rs if not r["is_null"]]
    return {
        "Retrieval — Recall@N": s["evidence_recall"] if False else
            sum(r["evidence_recall_at_N"] for r in ans) / len(ans),
        "Retrieval — Evidence Recall@k": s["evidence_recall"],
        "Retrieval — full-chain": s["full_chain_recall"],
        "Retrieval — nDCG@k": s["ndcg"],
        "Context — precision": s["context_precision"],
        "Context — citations resolve": sum(
            r["citation_resolvable"] for r in ans if r["citation_resolvable"] is not None
        ) / max(1, sum(1 for r in ans if r["citation_resolvable"] is not None)),
        "Answer — correctness": s["answer_correct"],
        "Answer — abstention recall": s["abstention_recall"],
    }

card = scorecard(rows)
tables.keyvalue([(k, f"{v:.3f}" if v is not None else "—") for k, v in card.items()],
                title="The baseline, read one lane at a time",
                kicker="Layered scorecard",
                caption="Retrieval is strong, context precision is where the distractors show "
                        "up, and the answer lane is where this offline reader is weakest. "
                        "One number would have hidden all three facts.")

### The metric multi-hop systems need

$$\text{EvidenceRecall@k} = \frac{|\text{gold evidence} \cap \text{top-}k|}{|\text{gold evidence}|}$$

That average is not enough. **Full-chain recall** — the fraction of questions where *every*
gold item is in top-k — is the number that predicts a correct answer, and it is always lower.


In [ ]:
by_hops = metrics.slice_report(answerable, by="hops",
                               keys=("evidence_recall", "full_chain_recall", "answer_correct"))
tables.show(by_hops, title="The gap between the average and the chain, by hop count",
            kicker="Sliced",
            caption="On single-hop questions the two are identical by definition. On two-hop "
                    "questions the gap is the whole story — and it is invisible in any "
                    "report that quotes only the average.",
            emphasize="full_chain_recall")

for by in ("question_type", "difficulty", "slice"):
    print(f"\n── sliced by {by} " + "─" * 40)
    print(metrics.slice_report(answerable, by=by).to_string(index=False))

---

## 6.2 Which metric catches which failure

Read this table right to left in a debugging session: pick the symptom you have, then the
metric that will move. The next cell does not just print the matrix — it **induces each
symptom** and checks that the predicted metric moves and the predicted metric stays flat.


In [ ]:
catalog.METRIC_SELECTION.show()

In [ ]:
def deltas(variant_rows, keys=("evidence_recall_at_N", "evidence_recall", "full_chain_recall",
                               "ndcg", "context_precision", "answer_correct")):
    a, b = metrics.summarize(rows), metrics.summarize(variant_rows)
    base_n = sum(r["evidence_recall_at_N"] for r in answerable) / len(answerable)
    var_ans = [r for r in variant_rows if not r["is_null"]]
    var_n = sum(r["evidence_recall_at_N"] for r in var_ans) / len(var_ans)
    out = {"Recall@N": var_n - base_n}
    for k in keys[1:]:
        if a.get(k) is not None and b.get(k) is not None:
            out[{"evidence_recall": "Evidence Recall@k", "full_chain_recall": "Full-chain",
                 "ndcg": "nDCG@k", "context_precision": "Context precision",
                 "answer_correct": "Answer correctness"}[k]] = b[k] - a[k]
    ab_a = metrics.abstention_scores(rows)["abstention_recall"] or 0
    ab_b = metrics.abstention_scores(variant_rows)["abstention_recall"] or 0
    out["Abstention recall"] = ab_b - ab_a
    return out


symptoms = {}
# 1 · A first-stage recall problem: narrow N hard.
symptoms["First-stage recall collapses (N=100 → 10)"] = deltas(
    pipeline.evaluate(pipe.variant("n10", n_candidates=10), bundle.questions, pipe.chunks,
                      personas=bundle.personas))
# 2 · A packing problem: same retrieval, fewer slots.
symptoms["Packing starves the chain (k=8 → 3)"] = deltas(
    pipeline.evaluate(pipe.variant("k3", k=3), bundle.questions, pipe.chunks,
                      personas=bundle.personas))
# 3 · A ranking problem: keep the pool, remove the reranker.
symptoms["Ranking degrades (reranker removed)"] = deltas(
    pipeline.evaluate(pipe.variant("norr", rerank="none"), bundle.questions, pipe.chunks,
                      personas=bundle.personas))
# 4 · A generation problem: same evidence, a reader that ignores it.
ungrounded = pipe.variant("ungrounded")
ungrounded.generator = generate.UngroundedGenerator()
symptoms["Generation ignores the evidence (fault injection)"] = deltas(
    pipeline.evaluate(ungrounded, bundle.questions, pipe.chunks, personas=bundle.personas))

frame = pd.DataFrame(symptoms).T.round(3)
tables.show(frame.reset_index().rename(columns={"index": "Induced symptom"}),
            title="Induce the symptom, watch which metric moves",
            kicker="The matrix, verified",
            caption="Each row is a real change to the running system. The columns that move "
                    "and the columns that stay flat are the diagnostic signature — this is "
                    "the metric-selection matrix earned rather than quoted.",
            emphasize="Induced symptom")

In [ ]:
tables.callout(
    "Look at the fault-injection row. <b>Every retrieval metric is unchanged</b> — same "
    "candidates, same packing, same context precision — and answer correctness collapses. "
    "That is the deck's line made concrete: <i>invented facts with correct evidence present</i> "
    "moves faithfulness and moves nothing in the retrieval lane. A team watching only "
    "end-to-end quality sees a regression and starts tuning the retriever.",
    kind="note", title="Why the retrieval lane must be read separately")

---

## 6.3 The judge, and why you cannot take its word for it

A rubric is a **release artefact**: judge model, temperature, rubric text and few-shot
examples are all versioned, and a change to any of them invalidates every score that came
before it.


In [ ]:
tables.show(pd.DataFrame([
    [r.name, r.version, r.fingerprint, r.criterion]
    for r in (judge.FAITHFULNESS, judge.COMPLETENESS, judge.CITATION)],
    columns=["Rubric", "Version", "Fingerprint", "The criterion, written as a procedure"]),
    title="Three rubrics, pinned",
    kicker="Release artefacts",
    caption="“Every factual claim maps to a span in a cited source” beats “is the answer "
            "faithful”. The first is a procedure two people can follow to the same verdict; "
            "the second is an adjective.",
    emphasize="Rubric")

In [ ]:
hj = judge.HeuristicJudge()

sample = [q for q in bundle.questions if q.question_type != "null"][:80]
verdicts, references = [], []
detail = []
for q in sample:
    tr = pipe.run(q.query, qid=q.qid, acl_groups=bundle.personas.get(q.persona))
    v = hj.judge_all(q.query, tr.answer, tr._packed_obj, q.answer)
    # The reference label is derived from gold: an answer is "good" when it is correct AND
    # the evidence chain that supports it actually reached the model.
    gm, _ = metrics.resolve_gold(q, pipe.chunks)
    ref = "pass" if (metrics.answer_correct(tr.answer, q.answer) >= 1.0
                     and metrics.full_chain_recall(tr.packed_ids, gm) == 1.0) else "fail"
    combined = "pass" if all(x.passed for x in v.values()) else "fail"
    verdicts.append(combined); references.append(ref)
    detail.append([q.qid, q.question_type, combined, ref,
                   "agree" if combined == ref else "DISAGREE",
                   v["faithfulness"].reasons[0][:52]])

rep = judge.calibration_report(verdicts, references, judge.FAITHFULNESS)
tables.keyvalue(list(rep.items()), title="Judge calibration against reference labels",
                kicker="Cohen's κ",
                caption="Raw agreement flatters a judge on a skewed set: one that always says "
                        "“pass” scores well and has learned nothing. κ corrects for chance.")

In [ ]:
tables.callout(
    "<b>Be precise about what was just measured.</b> Those reference labels are derived from "
    "gold evidence and gold answers, so they are correct by construction — which makes this "
    "an <i>accuracy</i> measurement dressed as a calibration."
    "<br><br>Real calibration has a step this cannot have: <b>two humans label the same "
    "100–200 examples against the same rubric, and where they disagree you fix the rubric, "
    "not the labels.</b> Their agreement with each other is the bar. If two humans agree only "
    "70% of the time, a judge at 72% is doing fine and your rubric is the problem — and you "
    "cannot know that without running the human pass. Budget a day for it; it is the cheapest "
    "day in the whole evaluation programme.", kind="warn", title="What this number is not")

In [ ]:
disagreements = [d for d in detail if d[4] == "DISAGREE"]
tables.show(pd.DataFrame(disagreements[:8],
                         columns=["qid", "type", "judge", "reference", "", "judge's reason"]
                         ).drop(columns=[""]),
            title=f"Where the judge and the reference disagree ({len(disagreements)} of "
                  f"{len(detail)})",
            kicker="Read the disagreements",
            caption="This is the step people skip. Every row is either a judge bug or an "
                    "ambiguous rubric, and you cannot tell which without reading them.",
            emphasize="judge")

### Attack the judge

Three probes. Each one takes a judge that just scored well and tries to make its verdict move
without changing anything that should matter.


In [ ]:
probe_q = sample[0]
tr = pipe.run(probe_q.query, qid=probe_q.qid)
short = tr.answer
padded = (short + " This is consistent with the broader picture described in the sources, "
          "and the supporting material provides substantial context for the conclusion, "
          "which follows directly from the evidence presented above.")

vb = judge.verbosity_probe(hj, probe_q.query, tr._packed_obj, short, padded)
print("VERBOSITY PROBE")
print(f"  short answer  : {len(short)} chars")
print(f"  padded answer : {len(padded)} chars (same claims, no new support)")
print(f"  verdict       : {vb['verdict']}")
if vb["moved"]:
    for k, (a, b) in vb["moved"].items():
        print(f"    {k}: {a} → {b}")

In [ ]:
from raglab import context as ctx_mod

cands = pipe.retriever.search(probe_q.query, pipe.cfg)
ranked = pipe.reranker.rerank(probe_q.query, cands, depth=pipe.cfg.rerank_depth)[:pipe.cfg.k]
packed_a = ctx_mod.build_prompt(probe_q.query, ranked, k=pipe.cfg.k)
packed_b = ctx_mod.build_prompt(probe_q.query, list(reversed(ranked)), k=pipe.cfg.k)

pb = judge.position_probe(hj, probe_q.query, packed_a, packed_b, tr.answer)
print("POSITION PROBE")
print(f"  same evidence set, reversed order")
print(f"  verdict: {pb['verdict']}")
if pb["moved"]:
    for k, (a, b) in pb["moved"].items():
        print(f"    {k}: {a} → {b}")
print("\n(The [S#] labels move with the reordering, so a citation-accuracy verdict that")
print(" changes here is correct behaviour rather than bias — which is exactly the kind of")
print(" distinction a probe forces you to think through before you trust its output.)")

In [ ]:
# JUDGE DRIFT: change the rubric, change nothing else, watch the scores move.
drift_rows = []
for threshold, label in ((0.35, "lenient  (support ≥ 0.35)"),
                         (0.50, "shipped  (support ≥ 0.50)"),
                         (0.70, "strict   (support ≥ 0.70)")):
    j = judge.HeuristicJudge(support_threshold=threshold)
    passes = 0
    for q in sample[:50]:
        t = pipe.run(q.query, qid=q.qid, acl_groups=bundle.personas.get(q.persona))
        v = j.judge_all(q.query, t.answer, t._packed_obj)
        passes += 1 if v["faithfulness"].passed else 0
    drift_rows.append([label, passes, f"{passes/50:.0%}"])

tables.show(pd.DataFrame(drift_rows, columns=["Rubric setting", "Faithfulness passes / 50",
                                              "Pass rate"]),
            title="Judge drift: the system did not change",
            kicker="Same answers, same evidence, same 50 questions",
            caption="One parameter inside the evaluator moves the headline quality number "
                    "across a range wider than most releases. An unexplained score jump with "
                    "no system change is judge drift until proven otherwise.",
            emphasize="Pass rate")

tables.show(pd.DataFrame([
    [name, desc, control] for name, desc, control in judge.JUDGE_BIASES],
    columns=["Bias", "What it looks like", "The control"]),
    title="Known biases to control for", kicker="Judge hygiene",
    source="Deck slide 68", emphasize="Bias")

---

## 6.4 The evaluation harness as a release gate

The point of this diagram is that the gate is **automated** and the override is **logged**.
Both matter, and the second one more than teams expect.


In [ ]:
viz.hld([
    dict(name="Trigger", tone="index", chain=False, nodes=[
        "Prompt change", "Retriever config", "Embedding / reranker version",
        "Chunking change", "Corpus reindex", "Nightly, unchanged code"]),
    dict(name="Run", tone="query", chain=False, nodes=[
        ("Retrieval suite", "deterministic, seconds, no model calls"),
        ("Answer suite", "exact match and schema checks where they are defensible"),
        ("Judge suite", "pinned judge model and rubric version"),
        ("Cost & latency", "tokens per query, p50/p95")]),
    dict(name="Gate", tone="warn", chain=False, nodes=[
        ("Hard block", "frozen-slice metric down beyond tolerance"),
        ("Hard block", "a previously-passing regression case now fails"),
        ("Warn", "cost per query up more than 15%"),
        ("Warn", "any single tenant or slice down while the average holds")]),
    dict(name="Ship", tone="control", nodes=[
        ("Canary a traffic slice", "compare against the control"),
        ("Roll forward or revert by config", "not by redeploy"),
        ("Feed failures back", "every human verdict becomes a regression case")]),
], title="The evaluation harness as a release gate", kicker="HLD",
   caption="The nightly run on unchanged code is not redundant. It is how you detect corpus "
           "drift, upstream model updates and judge drift — three things that change your "
           "system without anyone committing anything.",
   source="Deck slide 67")

In [ ]:
# Build the gate's inputs for a REAL candidate change, then run the tree.
candidate = pipe.variant("candidate: k=12, rerank depth 100", k=12, rerank_depth=100)

dev_q = [q for q in bundle.questions if q.slice == "dev"]
frozen_q = [q for q in bundle.questions if q.slice == "frozen"]

base_dev = [r for r in rows if r["slice"] == "dev"]
base_frozen = [r for r in rows if r["slice"] == "frozen"]
cand_dev = pipeline.evaluate(candidate, dev_q, pipe.chunks, personas=bundle.personas)
cand_frozen = pipeline.evaluate(candidate, frozen_q, pipe.chunks, personas=bundle.personas)

boot = metrics.paired_bootstrap(base_dev, cand_dev, "full_chain_recall")
noise_band = max(abs(boot["ci"][0]), abs(boot["ci"][1])) - abs(boot["delta"])

frozen_delta = (metrics.summarize(cand_frozen)["full_chain_recall"]
                - metrics.summarize(base_frozen)["full_chain_recall"])

# Per-slice check: did the average improve while a named slice got worse?
regressed = []
for qtype in ("inference", "comparison", "temporal"):
    a = [r for r in base_dev if r["question_type"] == qtype]
    b = [r for r in cand_dev if r["question_type"] == qtype]
    if a and b:
        d = (metrics.summarize(b)["full_chain_recall"]
             - metrics.summarize(a)["full_chain_recall"])
        if d < -0.02:
            regressed.append((qtype, round(d, 3)))

cost_a = metrics.summarize(base_dev)["cost_usd"]
cost_b = metrics.summarize(cand_dev)["cost_usd"]
cost_delta_pct = (cost_b - cost_a) / cost_a * 100

gate_ctx = {
    "frozen_drop": max(0.0, -frozen_delta),
    "tolerance": 0.02,
    "regressed_slices": regressed,
    "delta": boot["delta"],
    "noise_band": max(abs(boot["ci"][0]), abs(boot["ci"][1])),
    "cost_delta_pct": cost_delta_pct,
    "cost_envelope_pct": 15.0,
    "p95_ms": metrics.summarize(cand_dev).get("latency_ms_p95", 0),
    "p95_envelope_ms": 4000,
}
tables.keyvalue([
    ("Candidate change", candidate.name),
    ("Dev full-chain recall", f"{metrics.summarize(base_dev)['full_chain_recall']:.3f} → "
                              f"{metrics.summarize(cand_dev)['full_chain_recall']:.3f} "
                              f"({boot['delta']:+.3f})"),
    ("Paired bootstrap 95% CI", f"[{boot['ci'][0]:+.3f}, {boot['ci'][1]:+.3f}] → "
                                f"{boot['verdict']}"),
    ("Frozen slice", f"{frozen_delta:+.3f} (tolerance 0.02)"),
    ("Regressed slices", regressed or "none"),
    ("Cost per query", f"${cost_a:.5f} → ${cost_b:.5f} ({cost_delta_pct:+.1f}%)"),
], title="What the gate is about to read", kicker="Gate inputs",
   caption="Every field comes from a measurement above. None of it is an opinion, which is "
           "the point of writing the gate as a tree in the first place.")

In [ ]:
verdict = catalog.RELEASE_GATE.explain(gate_ctx)

In [ ]:
catalog.RELEASE_GATE.show_table()

---

## 6.5 Case study — DoorDash, and the two-layer pattern

Two quality layers with two different latency budgets, doing two different jobs.

**Inline, milliseconds.** A guardrail checks the drafted response against the retrieved
knowledge and against policy — grounding, compliance, tone — and blocks or regenerates rather
than shipping a bad answer. It must be cheap: it sits inside the user-facing latency budget
on every single turn.

**Offline, hours.** An LLM judge scores sampled transcripts on retrieval correctness, response
accuracy, coherence and policy adherence, then routes the failures back into prompt, retrieval
and knowledge-base fixes. It can afford to be expensive: it runs on a sample, off the request
path.

The team reported large reductions in hallucinated responses and compliance-related issues.
Equally important: the judge produced a *labelled failure stream*, which is what let the
knowledge base itself be fixed rather than just the prompt.


In [ ]:
from raglab import costs

inline = judge.HeuristicJudge()
import time
t0 = time.perf_counter()
blocked = 0
checked = 0
for q in sample[:40]:
    tr = pipe.run(q.query, qid=q.qid, acl_groups=bundle.personas.get(q.persona))
    v = inline.faithfulness(tr.answer, tr._packed_obj)
    checked += 1
    if not v.passed:
        blocked += 1
inline_ms = (time.perf_counter() - t0) / max(1, checked) * 1000

tables.show(pd.DataFrame([
    ["Inline guardrail", "every turn, before the reply is sent",
     f"{inline_ms:.1f} ms per check (heuristic, no model call)",
     f"blocked {blocked}/{checked} drafts", "Blocks. Must fit inside the user's latency budget."],
    ["Offline judge", "a sample of transcripts, off the request path",
     "seconds per conversation with a model judge",
     f"produced {len(disagreements)} labelled disagreements to read",
     "Teaches. Its output is the failure stream that fixes the knowledge base."],
], columns=["Layer", "When it runs", "Cost", "What it produced here", "Its job"]),
    title="Separate the thing that blocks from the thing that learns",
    kicker="Case study · DoorDash, publicly reported",
    caption="A single evaluator asked to do both ends up too slow to block and too shallow "
            "to teach. Ask in discovery: what is the cost of a wrong answer reaching this "
            "user? If it is regulatory, you need the inline layer on day one — not after the "
            "pilot.", source="Deck slide 71", emphasize="Layer")

---

## 6.6 Failure points and the interview


In [ ]:
lucky = [r for r in answerable if r["answer_correct"] >= 1.0 and r["evidence_recall"] == 0.0]
easy = [r for r in answerable if r["hops"] == 1]
hard = [r for r in answerable if r["hops"] >= 2]

tables.show(pd.DataFrame([
    ["Answer-only metric", f"{len(lucky)} of {len(answerable)} answers are correct with zero "
     "gold evidence retrieved",
     "The final answer is right by chance. An answer-only report counts these as passes and "
     "they will not survive a corpus refresh."],
    ["Static benchmark", "the frozen slice is 15% of the set and is never tuned against",
     "A retriever overfits the offline set and fails on new production terminology. The "
     "frozen slice plus the production-failure feed are the two defences."],
    ["Judge drift", f"pass rate moved from {drift_rows[0][2]} to {drift_rows[2][2]} on one "
     "rubric parameter",
     "A rubric prompt changes and silently shifts every score. Version the rubric and re-run "
     "the calibration set on every change."],
    ["Average masking", f"easy (1-hop) {metrics.summarize(easy)['full_chain_recall']:.3f} vs "
     f"hard (2-hop) {metrics.summarize(hard)['full_chain_recall']:.3f} full-chain",
     "Strong easy-case performance hides failure on long-tail multi-hop queries. Slice "
     "everything — it is the cheapest of the four fixes."],
], columns=["Signature", "Measured here", "Why it survives review"]),
    title="Failure points: evaluation systems", kicker="Failure points",
    source="Deck slide 70", emphasize="Signature")

In [ ]:
tables.show(pd.DataFrame([
    [catalog.SECTION_QUESTIONS[6][0],
     "Whether you have built one, or only used one",
     "The SEED → FILTER → MAINTAIN pipeline from notebook 02, with the human review sample "
     "and the frozen slice named as non-optional."],
    [catalog.SECTION_QUESTIONS[6][1],
     "Whether you can separate ranking quality from grounding",
     "Ranking: nDCG@k and context precision. Grounding: faithfulness with span-level "
     "checking. And know which one stays flat when the other moves — §6.2 is that table."],
    [catalog.SECTION_QUESTIONS[6][2],
     "Whether you know agreement statistics, not just accuracy",
     "Two humans label 100–200 examples against the same rubric; fix the rubric where they "
     "disagree; score the judge with Cohen's κ; compare judge–human agreement against "
     "human–human agreement; pin and version everything."],
    [catalog.SECTION_QUESTIONS[6][3],
     "Whether you have a real answer with a real threshold",
     "A frozen-slice drop beyond tolerance, or a previously-passing regression case that now "
     "fails. Warn on cost up more than 15% or a single slice down while the average holds. "
     "And say that the override is logged."],
], columns=["Question", "What the panel is testing", "What a strong answer covers"]),
    title="Typical interview questions: RAG evaluation",
    kicker="Section 6 · interview", source="Deck slide 72", emphasize="Question")

---

## 6.7 Checkpoint

1. Judged quality is up 6 points. Citation click-through and escalation rate have not moved.
   What do you believe?
2. Your judge agrees with human labels 72% of the time. Ship it or fix it?
3. A change improves the average and drops one tenant. The gate says block. Your stakeholder
   wants it shipped this week. What do you do?


In [ ]:
print("1 ·  The production signal. A judged gain with no movement in an independent signal is")
print("     judge drift until proven otherwise — re-score the calibration set against the")
print("     pinned rubric and check whether the judge, not the system, is what changed.\n")

print("2 ·  You cannot tell yet, and that is the answer the panel wants. 72% against WHAT?")
print("     If two humans agree 95% of the time, 72% is poor. If they agree 70%, the judge is")
print("     doing fine and the rubric is ambiguous. Measure human–human agreement first, then")
print("     compare, and track Cohen's κ over time as a metric in its own right.\n")

print("3 ·  Reproduce, find the mechanism, then make it a quantified choice rather than a")
print("     yes/no. Pull that tenant's real queries and confirm the regression appears in the")
print("     metrics; find the cause (identifier-heavy corpus plus a fusion shift, or short")
print("     documents plus a chunking change that moved avgdl); prefer a per-segment config")
print("     over an all-or-nothing ship. If you ship anyway, that owner hears it from you")
print("     first, with the number and a remediation date — and the override is logged.")

---

## What carries forward

- Three lanes, read separately. A regression in one lane and not the others has already told
  you which stage moved.
- Induce a symptom, watch which metric moves: that is how a metric-selection matrix gets
  earned rather than quoted.
- A judge is a component that can regress. Calibrate with κ against human labels, compare
  against human–human agreement, probe it for verbosity and position bias, and version every
  part of it.
- The gate reads measurements, not opinions — and a delta inside the noise band is not a
  result.

**Next:** `07_cost_and_token_optimization.ipynb` — four token categories, a prompt cache you
can break in one line, and one grounded answer priced to the cent.
